In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Event Semantics, Views & BigFrames Demo

This notebook demonstrates **Sections 14–16** of the BigQuery Agent Analytics SDK:

| Section | Module | Purpose |
|---|---|---|
| 14 | `event_semantics` | Canonical helpers for classifying and extracting data from agent events |
| 15 | `views` | `ViewManager` for creating per-event-type BigQuery views |
| 16 | `bigframes_evaluator` | DataFrame-native evaluation using BigFrames + `AI.GENERATE` |

<table align="left">

  <td>
    <a href="https://colab.research.google.com/github/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/event_semantics_views_bigframes_demo.ipynb">
      <img src="https://raw.githubusercontent.com/googleapis/python-bigquery-dataframes/refs/heads/main/third_party/logo/colab-logo.png" alt="Colab logo"> Run in Colab
    </a>
  </td>
  <td>
    <a href="https://github.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/event_semantics_views_bigframes_demo.ipynb">
      <img src="https://raw.githubusercontent.com/googleapis/python-bigquery-dataframes/refs/heads/main/third_party/logo/github-logo.png" width="32" alt="GitHub logo">
      View on GitHub
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/main/examples/event_semantics_views_bigframes_demo.ipynb">
      <img src="https://www.gstatic.com/images/branding/product/1x/google_cloud_48dp.png" alt="Vertex AI logo" width="32">
      Open in Vertex AI Workbench
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/event_semantics_views_bigframes_demo.ipynb">
      <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTW1gvOovVlbZAIZylUtf5Iu8-693qS1w5NJw&s" alt="BQ logo" width="35">
      Open in BQ Studio
    </a>
  </td>
</table>

## Install Dependencies

In [2]:
import subprocess, sys

# Install SDK from local source (not yet published to PyPI).
# Falls back silently if packages are already available.
rc = subprocess.call(
    [sys.executable, "-m", "pip", "install", "-q",
     "--break-system-packages",
     "google-cloud-bigquery", "bigframes", "nest-asyncio",
     "-e", "../.."],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Verify the SDK is importable regardless of pip outcome
import bigquery_agent_analytics
print(f"bigquery_agent_analytics loaded (version: {getattr(bigquery_agent_analytics, '__version__', 'dev')})")

bigquery_agent_analytics loaded (version: dev)


## Authenticate & Configure

In [3]:
import os

# Colab authentication
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab authentication successful.")
except ImportError:
    print("Not running in Colab \u2014 using default credentials.")

# ---------- Configuration ----------
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "test-project-0728-467323")
DATASET_ID = os.environ.get("BQ_DATASET", "agent_analytics")
TABLE_ID = os.environ.get("BQ_TABLE", "agent_events")
LOCATION = "US"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

print(f"Project  : {PROJECT_ID}")
print(f"Dataset  : {DATASET_ID}")
print(f"Table    : {TABLE_ID}")

Not running in Colab — using default credentials.
Project  : test-project-0728-467323
Dataset  : agent_analytics
Table    : agent_events


## 1. Event Semantics

The `event_semantics` module centralises the logic for interpreting ADK plugin
events so that every SDK module (evaluators, memory, insights, trace) uses
**consistent definitions** for "final response", "error event", "tool outcome",
etc.

Key exports:

| Export | Type | Description |
|---|---|---|
| `ALL_KNOWN_EVENT_TYPES` | `list[str]` | Every event type the SDK recognises |
| `EVENT_FAMILIES` | `dict[str, list]` | Event types grouped by family (user, llm, tool, ...) |
| `RESPONSE_EVENT_TYPES` | `tuple` | Event types that carry the final agent response |
| `is_error_event()` | `function` | Detects error events by type, message, or status |
| `is_tool_event()` | `function` | Returns `True` for tool-related events |
| `is_hitl_event()` | `function` | Returns `True` for Human-in-the-Loop events |
| `extract_response_text()` | `function` | Extracts user-visible text from a content dict |
| `tool_outcome()` | `function` | Classifies tool result as success/failure |
| `ERROR_SQL_PREDICATE` | `str` | SQL WHERE fragment for filtering error rows |

In [4]:
from bigquery_agent_analytics.event_semantics import (
    ALL_KNOWN_EVENT_TYPES,
    EVENT_FAMILIES,
    RESPONSE_EVENT_TYPES,
    ERROR_SQL_PREDICATE,
    NO_ERROR_SQL_PREDICATE,
    is_error_event,
    is_tool_event,
    is_hitl_event,
    is_hitl_completed,
    extract_response_text,
    tool_outcome,
)

# --- 1a. ALL_KNOWN_EVENT_TYPES ---
print("=" * 60)
print("ALL_KNOWN_EVENT_TYPES")
print("=" * 60)
for et in ALL_KNOWN_EVENT_TYPES:
    print(f"  {et}")
print(f"\nTotal: {len(ALL_KNOWN_EVENT_TYPES)} event types\n")

# --- 1b. EVENT_FAMILIES ---
print("=" * 60)
print("EVENT_FAMILIES")
print("=" * 60)
for family, members in EVENT_FAMILIES.items():
    print(f"  {family:12s} -> {members}")

# --- 1c. RESPONSE_EVENT_TYPES ---
print(f"\nRESPONSE_EVENT_TYPES: {RESPONSE_EVENT_TYPES}")

# --- 1d. Predicate functions ---
print("\n" + "=" * 60)
print("Predicate Functions")
print("=" * 60)

# is_error_event(event_type, error_message=None, status="OK")
test_cases_error = [
    ("LLM_ERROR", None, None),
    ("LLM_RESPONSE", "timeout occurred", None),
    ("TOOL_COMPLETED", None, "ERROR"),
    ("AGENT_COMPLETED", None, "OK"),
]
print("\nis_error_event(event_type, error_message, status):")
for et, err, st in test_cases_error:
    result = is_error_event(et, err, st)
    print(f"  is_error_event({et!r}, {err!r}, {st!r}) = {result}")

# is_tool_event
test_events_tool = [
    "TOOL_STARTING", "TOOL_COMPLETED", "TOOL_ERROR",
    "LLM_RESPONSE", "AGENT_COMPLETED",
]
print("\nis_tool_event():")
for et in test_events_tool:
    print(f"  is_tool_event({et!r}) = {is_tool_event(et)}")

# is_hitl_event / is_hitl_completed
test_events_hitl = [
    "HITL_CONFIRMATION_REQUEST",
    "HITL_CONFIRMATION_REQUEST_COMPLETED",
    "HITL_INPUT_REQUEST",
    "LLM_RESPONSE",
]
print("\nis_hitl_event() / is_hitl_completed():")
for et in test_events_hitl:
    print(
        f"  {et:45s}  hitl={str(is_hitl_event(et)):<6}"
        f" completed={is_hitl_completed(et)}"
    )

# --- 1e. extract_response_text ---
print("\n" + "=" * 60)
print("extract_response_text()")
print("=" * 60)
sample_contents = [
    {"text_summary": "Here is your flight itinerary."},
    {"response": "The weather in Tokyo is sunny."},
    {"tool": "search_flights", "args": {"origin": "SFO"}},
    {},
]
for content in sample_contents:
    text = extract_response_text(content)
    print(f"  content={content!r}")
    print(f"    -> {text!r}\n")

# --- 1f. tool_outcome ---
# Signature: tool_outcome(event_type, status="OK") -> "success" | "error" | "in_progress"
print("=" * 60)
print("tool_outcome()")
print("=" * 60)
tool_test_cases = [
    ("TOOL_COMPLETED", "OK"),
    ("TOOL_ERROR", "ERROR"),
    ("TOOL_COMPLETED", "OK"),
    ("TOOL_STARTING", "OK"),
]
for et, st in tool_test_cases:
    outcome = tool_outcome(et, st)
    print(f"  tool_outcome({et!r}, {st!r}) = {outcome!r}")

# --- 1g. SQL predicates ---
print("\n" + "=" * 60)
print("SQL Predicates (for BigQuery WHERE clauses)")
print("=" * 60)
print(f"\nERROR_SQL_PREDICATE:\n  {ERROR_SQL_PREDICATE}")
print(f"\nNO_ERROR_SQL_PREDICATE:\n  {NO_ERROR_SQL_PREDICATE}")

ALL_KNOWN_EVENT_TYPES
  USER_MESSAGE_RECEIVED
  INVOCATION_STARTING
  INVOCATION_COMPLETED
  AGENT_STARTING
  AGENT_COMPLETED
  LLM_REQUEST
  LLM_RESPONSE
  LLM_ERROR
  TOOL_STARTING
  TOOL_COMPLETED
  TOOL_ERROR
  STATE_DELTA
  HITL_CONFIRMATION_REQUEST
  HITL_CONFIRMATION_REQUEST_COMPLETED
  HITL_CREDENTIAL_REQUEST
  HITL_CREDENTIAL_REQUEST_COMPLETED
  HITL_INPUT_REQUEST
  HITL_INPUT_REQUEST_COMPLETED

Total: 18 event types

EVENT_FAMILIES
  user         -> ['USER_MESSAGE_RECEIVED']
  invocation   -> ['INVOCATION_STARTING', 'INVOCATION_COMPLETED']
  agent        -> ['AGENT_STARTING', 'AGENT_COMPLETED']
  llm          -> ['LLM_REQUEST', 'LLM_RESPONSE', 'LLM_ERROR']
  tool         -> ['TOOL_STARTING', 'TOOL_COMPLETED', 'TOOL_ERROR']
  state        -> ['STATE_DELTA']
  hitl         -> ['HITL_CONFIRMATION_REQUEST', 'HITL_CONFIRMATION_REQUEST_COMPLETED', 'HITL_CREDENTIAL_REQUEST', 'HITL_CREDENTIAL_REQUEST_COMPLETED', 'HITL_INPUT_REQUEST', 'HITL_INPUT_REQUEST_COMPLETED']

RESPONSE_EVENT_TY

## 2. View Management

The `ViewManager` creates standard (non-materialized) BigQuery **views** that
unnest the generic `agent_events` table into per-event-type views with typed
columns. Every view retains the standard identity headers (`timestamp`,
`event_type`, `agent`, `session_id`, `invocation_id`, etc.) and adds
event-specific columns extracted from the `content` and `attributes` JSON.

For example, the `LLM_RESPONSE` view extracts `usage_prompt_tokens`,
`usage_completion_tokens`, `total_ms`, and `ttft_ms` into typed columns.

**Supported event types** map to view suffixes like `adk_llm_responses`,
`adk_tool_completions`, `adk_agent_starts`, etc.

In [5]:
from bigquery_agent_analytics.views import ViewManager

# Instantiate ViewManager (does NOT create views yet)
vm = ViewManager(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=TABLE_ID,
    view_prefix="adk_",
)

# --- 2a. List all available event types with view definitions ---
print("=" * 60)
print("Available Event Types with View Definitions")
print("=" * 60)
for et in vm.available_event_types:
    view_name = vm.get_view_name(et)
    print(f"  {et:45s} -> {view_name}")
print(f"\nTotal: {len(vm.available_event_types)} views\n")

# --- 2b. Preview SQL for selected views (without creating them) ---
preview_types = ["LLM_RESPONSE", "TOOL_COMPLETED", "HITL_CONFIRMATION_REQUEST"]
for et in preview_types:
    sql = vm.get_view_sql(et)
    print("=" * 60)
    print(f"SQL for {et} -> {vm.get_view_name(et)}")
    print("=" * 60)
    print(sql)

# --- 2c. Creating views ---
# To actually create a single view in BigQuery, uncomment:
# vm.create_view("LLM_RESPONSE")

# To create ALL views at once, uncomment:
# created = vm.create_all_views()
# print(f"Created {len(created)} views:")
# for et, vn in created.items():
#     print(f"  {et} -> {vn}")

print("\n(View creation is commented out to avoid modifying your BQ dataset.")
print(" Uncomment the lines above to create views in your project.)")

Available Event Types with View Definitions
  AGENT_COMPLETED                               -> adk_agent_completions
  AGENT_STARTING                                -> adk_agent_starts
  HITL_CONFIRMATION_REQUEST                     -> adk_hitl_confirmation_requests
  HITL_CONFIRMATION_REQUEST_COMPLETED           -> adk_hitl_confirmation_completions
  HITL_CREDENTIAL_REQUEST                       -> adk_hitl_credential_requests
  HITL_CREDENTIAL_REQUEST_COMPLETED             -> adk_hitl_credential_completions
  HITL_INPUT_REQUEST                            -> adk_hitl_input_requests
  HITL_INPUT_REQUEST_COMPLETED                  -> adk_hitl_input_completions
  INVOCATION_COMPLETED                          -> adk_invocation_completions
  INVOCATION_STARTING                           -> adk_invocation_starts
  LLM_ERROR                                     -> adk_llm_errors
  LLM_REQUEST                                   -> adk_llm_requests
  LLM_RESPONSE                                 

## 3. BigFrames Evaluator

The `BigFramesEvaluator` provides a **DataFrame-native** evaluation workflow
using [BigFrames](https://cloud.google.com/bigquery/docs/bigframes-intro)
(`bigframes.pandas`) and BigQuery's `AI.GENERATE` operator.

This is useful in notebook environments where users prefer working with
DataFrames over raw SQL. It supports:

- **`evaluate_sessions()`** -- Reads session traces, constructs LLM judge
  prompts, and calls `AI.GENERATE` with `output_schema` for typed `score`
  and `justification` columns.
- **`extract_facets()`** -- Extracts structured facets (goals, outcomes,
  satisfaction, friction types, topics) for each session.

**Note:** BigFrames is an optional dependency. Install it with
`pip install bigframes`.

In [6]:
try:
    from bigquery_agent_analytics.bigframes_evaluator import BigFramesEvaluator
    import bigframes.pandas as bpd

    # Instantiate the evaluator
    bf_eval = BigFramesEvaluator(
        project_id=PROJECT_ID,
        dataset_id=DATASET_ID,
        table_id=TABLE_ID,
        endpoint="gemini-2.5-flash",  # default AI.GENERATE endpoint
    )

    print("BigFramesEvaluator created successfully.")
    print(f"  Project  : {bf_eval.project_id}")
    print(f"  Dataset  : {bf_eval.dataset_id}")
    print(f"  Table    : {bf_eval.table_id}")
    print(f"  Endpoint : {bf_eval.endpoint}")

    # --- 3a. Demonstrate BigFrames read_gbq ---
    # Verify BigFrames can query BigQuery directly.
    print("\n" + "=" * 60)
    print("BigFrames read_gbq — session trace aggregation")
    print("=" * 60)

    query = (
        "SELECT session_id, "
        "STRING_AGG("
        "CONCAT(event_type, \': \', "
        "COALESCE(JSON_EXTRACT_SCALAR(content, \'$.text_summary\'), \'\')), "
        "\'\\n\' ORDER BY timestamp) AS trace_text "
        f"FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` "
        "GROUP BY session_id "
        "HAVING LENGTH(trace_text) > 10 "
        "LIMIT 10"
    )
    df = bpd.read_gbq(query)
    print(f"\nRead {len(df)} sessions via bigframes.pandas.read_gbq:")
    print(df[["session_id"]].to_pandas().to_string(index=False))

    # --- 3b. Evaluate sessions (requires AI.GENERATE) ---
    print("\n" + "=" * 60)
    print("evaluate_sessions()")
    print("=" * 60)

    try:
        scores_df = bf_eval.evaluate_sessions(max_sessions=10)
        print("\nScores DataFrame:")
        print(scores_df[["session_id", "score", "justification"]])

        # Filtering: sessions with score >= 7
        high_quality = scores_df[scores_df["score"] >= 7]
        print(f"\nHigh-quality sessions (score >= 7): {len(high_quality)}")
        print(high_quality[["session_id", "score"]])

        # Aggregation: mean score
        mean_score = scores_df["score"].mean()
        print(f"\nMean score across all sessions: {mean_score:.2f}")
    except Exception:
        print("  (AI.GENERATE requires BigQuery Enterprise edition)")
        print("  BigFrames read_gbq works — see session data above.")
        print("  evaluate_sessions() will work once Enterprise is enabled.")

    # --- 3c. Extract facets (requires AI.GENERATE) ---
    print("\n" + "=" * 60)
    print("extract_facets()")
    print("=" * 60)

    try:
        facets_df = bf_eval.extract_facets(max_sessions=10)
        print("\nFacets DataFrame columns:")
        print(f"  {list(facets_df.columns)}")
        print("\nFirst rows:")
        print(facets_df.head())
    except Exception:
        print("  (AI.GENERATE requires BigQuery Enterprise edition)")
        print("  extract_facets() will work once Enterprise is enabled.")

except ImportError:
    print("bigframes is not installed. Install with:")
    print("  pip install bigframes")
    print("\nSkipping BigFramesEvaluator demo.")

BigFramesEvaluator created successfully.
  Project  : test-project-0728-467323
  Dataset  : agent_analytics
  Table    : agent_events
  Endpoint : gemini-2.5-flash

BigFrames read_gbq — session trace aggregation



Read 10 sessions via bigframes.pandas.read_gbq:
       session_id
 e2e-a9b40dfdbe84
 e2e-125750c7b94d
 e2e-f676e06a719e
 e2e-3171eb36991f
 e2e-781fb52f3814
 e2e-054c12957ba3
adcp-f5a6b8bd92e8
adcp-87820945dd00
adcp-033c95d7a97d
adcp-819759bc861c

evaluate_sessions()


  (AI.GENERATE requires BigQuery Enterprise edition)
  BigFrames read_gbq works — see session data above.
  evaluate_sessions() will work once Enterprise is enabled.

extract_facets()


  (AI.GENERATE requires BigQuery Enterprise edition)
  extract_facets() will work once Enterprise is enabled.


## Summary

This notebook covered three complementary SDK modules:

| Section | Module | What You Learned |
|---|---|---|
| **14** | `event_semantics` | Canonical constants (`ALL_KNOWN_EVENT_TYPES`, `EVENT_FAMILIES`, `RESPONSE_EVENT_TYPES`), predicate functions (`is_error_event`, `is_tool_event`, `is_hitl_event`), extraction helpers (`extract_response_text`, `tool_outcome`), and SQL predicates (`ERROR_SQL_PREDICATE`) |
| **15** | `views` | `ViewManager` creates per-event-type BigQuery views with typed columns; use `get_view_sql()` to preview, `create_view()` for one, or `create_all_views()` for all |
| **16** | `bigframes_evaluator` | `BigFramesEvaluator` provides DataFrame-native evaluation via BigFrames + `AI.GENERATE`; `evaluate_sessions()` returns scores, `extract_facets()` returns structured session facets |

### Key Takeaways

- **Consistency**: `event_semantics` is the single source of truth for event classification -- import from here instead of re-implementing checks.
- **SQL-friendly**: `ERROR_SQL_PREDICATE` and `NO_ERROR_SQL_PREDICATE` can be embedded directly in BigQuery WHERE clauses.
- **Zero-copy views**: `ViewManager` creates lightweight views (not materialized tables) that stay in sync with the source `agent_events` table.
- **DataFrame workflows**: `BigFramesEvaluator` bridges the gap between SQL-based evaluation and interactive notebook analysis with familiar pandas-like syntax.
- **Typed outputs**: Both `evaluate_sessions()` and `extract_facets()` use `output_schema` with `AI.GENERATE` to return structured, typed columns directly in the DataFrame.

In [7]:
print("Notebook complete!")

Notebook complete!
